# Optical Character Recognition (OCR)

## Installation

Mac: 
- `brew install tesseract`

Windows: 
- https://github.com/tesseract-ocr/tesseract/releases/download/5.5.0/tesseract-ocr-w64-setup-5.5.0.20241111.exe
- Add to your script: `pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"`

> **Optional side track.** OCR matters for scanned documents. The core RAG path (6.5 onward) does not depend on this notebook; it needs Tesseract installed (`brew install tesseract`).


## What is OCR?

**Optical Character Recognition (OCR)** is the technology that converts images containing typed, handwritten, or printed text into machine-encoded text. It bridges the gap between the physical and digital worlds.

### Brief History
| Era | Milestone |
|-----|-----------|
| 1914 | Emanuel Goldberg builds a machine that reads characters and converts to telegraph code |
| 1950s | First commercial OCR machines for reading bank checks |
| 1974 | Ray Kurzweil creates the first omni-font OCR (Kurzweil Reading Machine) |
| 1995 | Tesseract open-sourced by HP, later adopted by Google |
| 2010s | Deep learning (CNNs, LSTMs) dramatically improves accuracy |
| 2020s | Transformer-based models (TrOCR) push state-of-the-art further |

### How OCR Works — The Pipeline
```
Image Input
    ↓
Preprocessing     (grayscale, denoise, deskew, thresholding)
    ↓
Layout Analysis   (detect columns, paragraphs, text regions)
    ↓
Character/Word Segmentation
    ↓
Feature Extraction (classical: strokes/projections; DL: CNN features)
    ↓
Recognition       (HMM, LSTM, or Transformer)
    ↓
Post-processing   (dictionary lookup, language model correction)
    ↓
Structured Text Output
```

### Common Use Cases
- **Document digitization** — scanning books, legal contracts, invoices
- **Receipt parsing** — expense management apps (e.g., Expensify)
- **License plate recognition** — traffic cameras, parking systems
- **Passport / ID verification** — KYC in banking and fintech
- **Accessibility** — reading printed text aloud for visually impaired users
- **Medical records** — digitizing handwritten doctor notes
- **Automated data entry** — forms, surveys, tax documents
- **Historical archive processing** — digitizing old newspapers

In [ ]:
%pip install -q pytesseract pillow opencv-python-headless matplotlib pandas

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import numpy as np
import shutil, os
import pytesseract

# Configure Tesseract binary path
tesseract_bin = shutil.which('tesseract')
if tesseract_bin:
    pytesseract.pytesseract.tesseract_cmd = tesseract_bin
    print(f'Tesseract executable found at: {tesseract_bin}')
else:
    raise FileNotFoundError(
        "Tesseract binary not found in PATH. Install via 'brew install tesseract' (macOS) "
        "or 'apt install tesseract-ocr' (Linux)."
    )


---
## Section 1: Basic OCR with Tesseract

[**Tesseract**](https://github.com/tesseract-ocr/tesseract) is the most widely used open-source OCR engine, originally developed by HP and now maintained by Google.  
`pytesseract` is a Python wrapper around the Tesseract CLI.

Below we create a synthetic receipt image and run basic OCR on it.

In [ ]:
import os
from PIL import Image, ImageDraw

os.makedirs('data', exist_ok=True)

def make_receipt(path):
    img = Image.new('RGB', (400, 500), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    lines = [
        '      CORNER BAKERY CAFE',
        '    123 Main St, Suite 4B',
        '   Tel: (555) 867-5309',
        '─' * 38,
        'Cappuccino Large         $5.25',
        'Avocado Toast            $9.50',
        'Blueberry Muffin         $3.75',
        '─' * 38,
        'Subtotal                $18.50',
        'Tax (8.5%)               $1.57',
        '─' * 38,
        'TOTAL                   $20.07',
        '',
        '  Thank you for your visit!',
        '  Please come again  :)',
    ]
    y = 20
    for line in lines:
        draw.text((10, y), line, fill=(0, 0, 0))
        y += 28
    img.save(path)
    return img

img = make_receipt('data/receipt1.jpg')

# Display image
plt.figure(figsize=(4, 5))
plt.imshow(img)
plt.axis('off')
plt.title('Synthetic Receipt Image')
plt.show()

# Run Tesseract OCR on image
text = pytesseract.image_to_string(img)

print('Recognized Text:\n')
print(text)


---
## Section 2: Image Preprocessing — The Key to Good OCR

Raw images are rarely OCR-ready. Preprocessing can dramatically improve accuracy:

| Step | Why it helps |
|------|-------------|
| **Grayscale conversion** | Removes color noise; most OCR engines work on grayscale internally |
| **Binarization (thresholding)** | Separates text (black) from background (white) |
| **Adaptive thresholding** | Handles uneven lighting across the image |
| **Denoising** | Removes salt-and-pepper noise |
| **Deskewing** | Corrects tilted/rotated scans |
| **Scaling / upsampling** | Tesseract works best at ≥300 DPI; small text benefits from upscaling |
| **Border removal** | Removes scan artifacts near page edges |

In [ ]:
import numpy as np
import cv2

# ── Create a noisy test image ────────────────────────────────
def make_noisy_image(path):
    img = Image.new('RGB', (500, 100), color=(240, 240, 240))
    draw = ImageDraw.Draw(img)
    draw.text((20, 35), 'INVOICE #94821 - PAID', fill=(30, 30, 30))
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, 25, arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    noisy_img = Image.fromarray(arr)
    noisy_img.save(path)
    return noisy_img

noisy = make_noisy_image('data/noisy_invoice.png')

# ── Preprocessing Pipeline (Grayscale, Gaussian Blur, Otsu Binarization) ──────
img_bgr = np.array(noisy)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_RGB2GRAY)
blurred = cv2.GaussianBlur(gray, (3, 3), 0)
_, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].imshow(noisy); axes[0].set_title('1. Original Noisy'); axes[0].axis('off')
axes[1].imshow(gray, cmap='gray'); axes[1].set_title('2. Grayscale'); axes[1].axis('off')
axes[2].imshow(thresh, cmap='gray'); axes[2].set_title('3. Binarized (Otsu)'); axes[2].axis('off')
plt.tight_layout(); plt.show()


---
## Section 3: Tesseract Configuration

Tesseract exposes powerful knobs via the `config` parameter.

### Page Segmentation Modes (PSM)
Controls how Tesseract segments the image before recognition.

| PSM | Description | Best for |
|-----|-------------|----------|
| 0 | Orientation and Script Detection (OSD) only | Pre-step |
| 3 | Fully automatic page segmentation *(default)* | Multi-paragraph docs |
| 4 | Single column of text | Newspaper columns |
| 6 | Assume a single uniform block of text | Paragraphs |
| 7 | Single text line | One-line labels |
| 8 | Single word | Captchas, license plates |
| 10 | Single character | Individual letters |
| 11 | Sparse text — find text anywhere | Receipts, forms |
| 13 | Raw line (no word joining heuristics) | Debug |

### OCR Engine Modes (OEM)
| OEM | Description |
|-----|-------------|
| 0 | Legacy Tesseract engine only |
| 1 | Neural LSTM engine only *(recommended)* |
| 2 | Legacy + LSTM combined |
| 3 | Default — pick what's available |

In [ ]:
# ── Create a single-line label image ─────────────────────────────────────────
label_img = Image.new('RGB', (360, 60), color=(255, 255, 255))
draw = ImageDraw.Draw(label_img)
draw.text((10, 15), 'FRAGILE - HANDLE WITH CARE', fill=(0, 0, 0))

# ── Compare PSM 3 (auto) vs PSM 7 (single text line) ─────────────────────────
configs = [
    ('--psm 3',  'PSM 3 : Fully automatic page segmentation'),
    ('--psm 7',  'PSM 7 : Treat image as a single text line'),
    ('--psm 8',  'PSM 8 : Treat image as a single word'),
    ('--psm 11', 'PSM 11: Sparse text with OSD'),
]

print(f"{'Config':<12} {'Output'}")
print('-' * 45)
for cfg, desc in configs:
    out = pytesseract.image_to_string(label_img, config=cfg).strip()
    print(f"{cfg:<12} '{out}'")


---
## Section 4: Extracting Structured Data — Bounding Boxes

Beyond plain text, Tesseract can return **word-level position data** — bounding boxes, confidence scores, block/paragraph/line structure. This is essential for:

- Highlighting where text was found in the image
- Building form parsers that know which value belongs to which label
- Filtering low-confidence recognitions

The key functions are:

| Function | Returns |
|----------|---------|
| `image_to_string()` | Plain string |
| `image_to_data()` | DataFrame with box coords + confidence |
| `image_to_boxes()` | Character-level bounding boxes |
| `image_to_osd()` | Orientation / script detection |
| `image_to_pdf_or_hocr()` | hOCR XML or searchable PDF |

In [ ]:
import pandas as pd

receipt_img = Image.open('data/receipt1.jpg')

# Extract word-level position and confidence data using Tesseract
raw_data = pytesseract.image_to_data(receipt_img, output_type=pytesseract.Output.DATAFRAME)
data_clean = raw_data[raw_data.text.notnull() & (raw_data.text.str.strip() != '')].copy()
data_clean['conf'] = pd.to_numeric(data_clean['conf'], errors='coerce')
data_clean = data_clean[data_clean['conf'] > 0]

print(f'Detected {len(data_clean)} word boxes:')
print(data_clean[['text', 'conf', 'left', 'top', 'width', 'height']].head(6))


In [ ]:
# ── Draw bounding boxes on the image ─────────────────────────────────────────
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(receipt_img)
ax.axis("off")
ax.set_title("Word-level bounding boxes (conf ≥ 60)")

for _, row in data_clean[data_clean["conf"] >= 60].iterrows():
    x, y, w, h = row["left"], row["top"], row["width"], row["height"]
    rect = mpatches.Rectangle((x, y), w, h, linewidth=1,
                               edgecolor="lime", facecolor="none")
    ax.add_patch(rect)
    ax.text(x, y - 2, f"{int(row['conf'])}%", fontsize=5, color="red")

plt.tight_layout()
plt.show()

In [ ]:
# ── Confidence distribution ───────────────────────────────────────────────────
plt.figure(figsize=(7, 3))
plt.hist(data_clean["conf"], bins=20, color="steelblue", edgecolor="white")
plt.xlabel("Confidence (%)")
plt.ylabel("Word count")
plt.title("Tesseract Per-word Confidence Distribution")
plt.axvline(60, color="red", linestyle="--", label="threshold = 60")
plt.legend()
plt.tight_layout()
plt.show()

low_conf = data_clean[data_clean["conf"] < 60]
print(f"Low-confidence words (< 60): {len(low_conf)}")
print(low_conf[["text", "conf"]].to_string(index=False))

## Section 5: Post-processing OCR Output

Raw OCR text is noisy. Post-processing steps include:

1. **Regex-based extraction** — pull out specific fields (dates, prices, phone numbers)
2. **Spell correction** — fix common OCR mis-reads (`0` vs `O`, `1` vs `l`)
3. **NLP parsing** — POS tagging, NER to label entities
4. **Layout reconstruction** — use bounding-box coordinates to reassemble tabular data

In [ ]:
import re

raw = pytesseract.image_to_string(receipt_img, config='--oem 1 --psm 11')

print('Raw OCR output:\n')
print(raw)

# ── Post-processing with Regex ────────────────────────────────────────────────
print('=' * 40)
print('Extracted Structured Entities:')
prices = re.findall(r'\$\s*\d+\.\d{2}', raw)
print(f'Price tags found : {prices}')
total_match = re.search(r'TOTAL\s+(\$\d+\.\d{2})', raw)
if total_match:
    print(f'Receipt Total    : {total_match.group(1)}')


In [ ]:
# Layout reconstruction from bounding boxes
data_clean['line_group'] = (data_clean['top'] // 25) * 25
lines = []
for line_y, group in data_clean.groupby('line_group'):
    sorted_words = group.sort_values('left')
    lines.append(' '.join(sorted_words['text']))

print('Reconstructed Receipt Lines:')
print('-' * 40)
for l in lines:
    print(l)


---
## Section 5b: Template-Based Field Extraction (Receipts)

Once OCR has produced text, **regex templates** turn it into structured fields. We use clean sample receipt text here so the patterns are easy to read. For entity-based extraction with spaCy, see the NER section of `6.2-Ingestion-PDFs.ipynb`.

In [ ]:
import re
import pprint
from collections import defaultdict

In [ ]:
# Sample receipt text
receipt_text = """
Receipt Number: 12345
Store: ABC Store
Date: 2025-02-17
Items:
- Item 1: 2 x $5.00
- Item 2: 1 x $2.50
Total: $12.50
"""

In [ ]:
# Define a pattern template to extract various pieces of data
patterns = {
    "receipt_number": r"Receipt Number: (\d+)",
    "store": r"Store: ([\w\s]+)",
    "date": r"Date: (\d{4}-\d{2}-\d{2})",
    "items": r"- ([\w\s]+): (\d+) x \$(\d+\.\d{2})",
    "total": r"Total: \$(\d+\.\d{2})"
}

In [ ]:
# Create a dictionary to store the extracted data
extracted_data = {}

# Extract the data using regex
for key, pattern in patterns.items():
    match = re.search(pattern, receipt_text)
    if match:
        if key == "items":
            # For items, create a list of dictionaries with item names, quantities, and prices
            items = []
            item_matches = re.findall(pattern, receipt_text)
            for item in item_matches:
                items.append({"item": item[0], "quantity": int(item[1]), "price": float(item[2])})
            extracted_data[key] = items
        else:
            extracted_data[key] = match.group(1)

# Display the extracted data
pprint.pp(extracted_data)

In [ ]:
# Sample receipt with noise and optional fields
receipt_text = """
Receipt Number: 12345
Store: ABC Supermarket
Date: 2025-02-17
Order Number: 67890
Receipt Totals: $15.00
Item Details:
- Product: 3 x $2.00
- Item: 2 x $1.50
Total Due: $15.00
"""

In [ ]:
# Define regex patterns for flexible matching
patterns = {
    "receipt_number": r"Receipt Number[:\-]?\s*(\d+)",
    "store": r"Store[:\-]?\s*([\w\s]+)",
    "date": r"Date[:\-]?\s*(\d{4}-\d{2}-\d{2})",
    "items": r"[-\*]\s*(\w[\w\s]*):\s*(\d+)\s*x\s*\$(\d+\.\d{2})",  # Handles 'Product' or 'Item'
    "total": r"(?:Total|Receipt Totals|Total Due)[:\-]?\s*\$(\d+\.\d{2})"
}

# Create a dictionary to store extracted data
extracted_data = defaultdict(list)

# Function to extract items using context (for more accurate extraction)
def extract_items_with_context(text):
    items = []
    for line in text.splitlines():   # one receipt line at a time
        if "x" in line and "$" in line:
            # Extract item-related details            
            matches = re.findall(r"([A-Za-z\s]+):\s*(\d+)\s*x\s*\$(\d+\.\d{2})", line)
            if matches:
                for match in matches:
                    item = {"item": match[0].strip(), "quantity": int(match[1]), "price": float(match[2])}
                    items.append(item)
    return items

# Function to apply regex patterns
def extract_with_regex(patterns, text):
    data = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, text)
        if match:
            if key == "items":
                items = extract_items_with_context(text)
                data[key] = items
            else:
                data[key] = match.group(1)
    return data

# Extract the data using regex and context-based item extraction
extracted_data = extract_with_regex(patterns, receipt_text)

# Display the extracted data
pprint.pp(extracted_data)

---
## Section 6: Modern Vision-Language Models (VLMs) vs. Classical OCR

Classical OCR (Tesseract) follows a **two-stage cascade**:
$$\text{Image} \xrightarrow{\text{OCR Engine}} \text{Raw Bounding Boxes} \xrightarrow{\text{Regex / NLP Heuristics}} \text{Structured Output}$$

Modern **Multimodal Vision-Language Models** (e.g. **PaliGemma**, **Qwen2-VL**, **GPT-4o Vision**, **LLaVA**) replace this fragile pipeline with **end-to-end multimodal autoregression**:

```
Input Document Image
         │
         ▼
  [Vision Encoder (SigLIP / ViT)] ──▶ Visual Tokens [v_1, v_2, ..., v_k]
                                                 │
  Text Prompt: "Extract total and items as JSON" ┼──▶ [Multimodal LLM Decoder] ──▶ Structured JSON
                                                 │
```

### Advantages of End-to-End VLMs
1. **Context-Aware Visual Reasoning**: VLMs understand layout, headers, footnotes, and handwritten signatures simultaneously without heuristic coordinate grouping.
2. **Direct Schema Adherence**: Instruct the model to return valid JSON matching a Pydantic schema in a single zero-shot pass.
3. **Robustness to Visual Artifacts**: Tolerates complex multi-column layouts, watermarks, stamps, and tilted orientation without manual deskewing.


In [ ]:
# Multimodal Document Ingestion: Formatting Images for Vision-Language Models (VLMs)
import base64
import io

def image_to_base64_data_uri(img: Image.Image, format: str = 'JPEG') -> str:
    """Converts a PIL Image to a base64 data URI accepted by OpenAI/OpenRouter/VLM vision APIs."""
    buf = io.BytesIO()
    img.save(buf, format=format)
    encoded = base64.b64encode(buf.getvalue()).decode('utf-8')
    return f'data:image/{format.lower()};base64,{encoded}'

# Build standard multimodal payload for receipt understanding
receipt_img = Image.open('data/receipt1.jpg')
data_uri = image_to_base64_data_uri(receipt_img)

vlm_prompt = [
    {
        'role': 'user',
        'content': [
            {'type': 'text', 'text': 'Extract the store name, date, itemized list, and total amount as JSON.'},
            {'type': 'image_url', 'image_url': {'url': data_uri[:100] + '... [truncated for display]'}}
        ]
    }
]

print('VLM Multimodal Payload Schema:')
print(f"Role     : {vlm_prompt[0]['role']}")
print(f"Text Goal: {vlm_prompt[0]['content'][0]['text']}")
print(f"Image URI: {vlm_prompt[0]['content'][1]['image_url']['url']}")
print(f"Image dimensions: {receipt_img.size} (Width x Height)")


---
## Section 7: Limitations of OCR

Understanding where OCR breaks down is just as important as knowing what it can do.

### Accuracy Killers
| Problem | Impact | Mitigation |
|---------|--------|------------|
| **Low resolution** (< 150 DPI) | Characters unrecognizable | Upsample before OCR |
| **Handwriting** | Very high error rate | Use a handwriting-specific model (e.g., Google Vision, AWS Textract) |
| **Skewed / rotated text** | Words split incorrectly | Deskew with OpenCV before OCR |
| **Fonts with ligatures** | `fi`, `fl`, `ffi` merged incorrectly | Post-process / custom traineddata |
| **Low contrast** | Background bleeds into text | Adaptive thresholding |
| **Complex layouts** | Mixed columns + images | Use layout detection (LayoutParser / PaddleOCR) |
| **Mathematical formulas** | Symbols misread | Use specialized engines (Mathpix, LaTeX-OCR) |
| **Multi-language in one doc** | Wrong charset applied | Specify all languages: `lang="eng+fra"` |
| **Decorative / artistic fonts** | Very high error rate | DL engines help but not perfect |

### Accuracy Benchmarks (approximate)
| Engine | Printed text | Handwriting | Scene text |
|--------|-------------|-------------|------------|
| Tesseract 5 (LSTM) | 95–98% | ~60% | ~70% |
| EasyOCR | 93–97% | ~65% | ~85% |
| Google Vision API | 98–99% | ~80% | ~95% |
| AWS Textract | 98–99% | ~75% | ~90% |
| TrOCR (Microsoft) | ~99% | ~90% | ~92% |

> Accuracy depends heavily on image quality and language. Always validate on your specific domain.

### When to Use What
- **Tesseract** — offline, free, no data leaves your machine, good printed text
- **EasyOCR** — better scene text, 80 languages, free
- **PaddleOCR** — excellent multilingual, free, includes layout detection
- **Cloud APIs (Google/AWS/Azure)** — highest accuracy, pay-per-use, best for handwriting
- **TrOCR** — state-of-the-art for printed/handwritten, runs locally via HuggingFace

## Summary & Key Takeaways

1. **OCR as Multimodal Ingestion**: Converts raw visual document images (scanned PDFs, receipts, forms) into machine-readable text tokens for downstream NLP and RAG pipelines.
2. **Image Preprocessing is Critical**: Binarization (Otsu thresholding), deskewing (Hough transforms), and contrast enhancement (CLAHE) dramatically reduce character error rates.
3. **Tesseract Engine Knobs**: Page Segmentation Modes (PSM 6 for uniform blocks, PSM 11 for sparse text) control structural parsing before character recognition.
4. **Bounding Box Spatial Layout**: Extracting coordinate geometries allows reconstructing tables, key-value pairs, and column flows.
5. **Template Extraction**: Regex templates pull receipt number, store, date, items and total out of OCR text, but break when the layout changes, which is the case for VLMs.
6. **Paradigm Shift to VLMs**: End-to-end Vision-Language Models (PaliGemma, Qwen2-VL, GPT-4o) replace brittle 2-stage OCR pipelines with unified multimodal cross-attention and direct structured JSON generation.
